In [1]:
# Preparazione e controllo dati

# Importiamo le librerie per leggere e controllare i dati.
import pandas as pd
import numpy as np
from pathlib import Path

# Apriamo i due CSV separatamente: rappresentano gli stessi noleggi con dettaglio diverso.
hour = pd.read_csv('../data/raw/hour.csv')
day = pd.read_csv('../data/raw/day.csv')

# Controlliamo righe, colonne, valori mancanti e duplicati.
for name, df in [('hour', hour), ('day', day)]:
    print(f'{name}: {len(df):,} rows, {len(df.columns)} columns')
    print(' Missing values:', int(df.isna().sum().sum()),
          '| Duplicate full rows:', int(df.duplicated().sum()))
    print(' Unique dates:', df['dteday'].nunique())

# Cerchiamo date e ore ripetute, che non dovrebbero esserci.
print('Duplicate (date, hour) keys:', int(hour.duplicated(['dteday', 'hr']).sum()))
print('Duplicate daily dates:', int(day.duplicated('dteday').sum()))

# Guardiamo alcune righe per riconoscere le colonne.
display(hour.head(3))

hour: 17,379 rows, 17 columns
 Missing values: 0 | Duplicate full rows: 0
 Unique dates: 731
day: 731 rows, 16 columns
 Missing values: 0 | Duplicate full rows: 0
 Unique dates: 731
Duplicate (date, hour) keys: 0
Duplicate daily dates: 0


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32


In [2]:
# Convertiamo le date e aggiungiamo anno, mese e giorno della settimana.
for df in [hour, day]:
    df['dteday'] = pd.to_datetime(df['dteday'])
    df['year'] = df['dteday'].dt.year
    df['month_name'] = df['dteday'].dt.month_name()
    df['day_name'] = df['dteday'].dt.day_name()

# Creiamo una data completa di ora per i grafici temporali.
hour['datetime'] = hour['dteday'] + pd.to_timedelta(hour['hr'], unit='h')

# Traduciamo i codici del meteo in etichette comprensibili.
weather_labels = {
    1: 'Clear / Partly Cloudy',
    2: 'Mist / Cloudy',
    3: 'Light Rain / Snow',
    4: 'Severe Weather'
}

for df in [hour, day]:
    df['weather_label'] = df['weathersit'].map(weather_labels)

    # Convertiamo temperatura e umidità nelle unità originali del dataset.
    df['temperature_c'] = df['temp'] * 41
    df['feels_like_c'] = df['atemp'] * 50
    df['humidity_pct'] = df['hum'] * 100

# Controlliamo un esempio dei dati trasformati.
display(hour[['datetime','temperature_c','humidity_pct','weather_label','cnt']].head())

,datetime,temperature_c,humidity_pct,weather_label,cnt
0,2011-01-01 00:00:00,9.84,81.0,Clear / Partly Cloudy,16
1,2011-01-01 01:00:00,9.02,80.0,Clear / Partly Cloudy,40
2,2011-01-01 02:00:00,9.02,80.0,Clear / Partly Cloudy,32
3,2011-01-01 03:00:00,9.84,75.0,Clear / Partly Cloudy,13
4,2011-01-01 04:00:00,9.84,75.0,Clear / Partly Cloudy,1


In [3]:
# Verifichiamo che occasionali + registrati corrisponda al totale.
assert (hour['casual'] + hour['registered'] == hour['cnt']).all()
assert (day['casual'] + day['registered'] == day['cnt']).all()

# Controlliamo che la somma dei noleggi orari sia uguale al totale giornaliero.
daily_from_hour = hour.groupby('dteday')['cnt'].sum().sort_index()
daily_from_day = day.set_index('dteday')['cnt'].sort_index()
assert daily_from_hour.equals(daily_from_day)

# Costruiamo una tabella con 24 ore per giorno per trovare le ore assenti.
hourly_grid = hour.pivot_table(index='dteday', columns='hr', values='cnt', aggfunc='sum')
hourly_grid = hourly_grid.reindex(columns=range(24))

# Contiamo le ore mancanti e i giorni incompleti senza inventare valori.
missing_hours = int(hourly_grid.isna().sum().sum())
incomplete_days = int(hourly_grid.isna().any(axis=1).sum())

# Stampiamo il riepilogo dei controlli.
print('Hourly missing cells in date x 24-hour grid:', missing_hours)
print('Days with at least one absent hourly record:', incomplete_days)
print('All rental totals consistent: True')
print('Weather code frequencies:')
display(hour['weather_label'].value_counts().to_frame('Observed hours'))

Hourly missing cells in date x 24-hour grid: 165
Days with at least one absent hourly record: 76
All rental totals consistent: True
Weather code frequencies:


,Observed hours
weather_label,
Clear / Partly Cloudy,11413
Mist / Cloudy,4544
Light Rain / Snow,1419
Severe Weather,3


In [4]:
# Creiamo la cartella di destinazione e salviamo i due CSV puliti.
Path('../data/processed').mkdir(parents=True, exist_ok=True)
hour.to_csv('../data/processed/bike_hour_clean.csv', index=False)
day.to_csv('../data/processed/bike_day_clean.csv', index=False)
print('Saved bike_hour_clean.csv and bike_day_clean.csv.')

Saved bike_hour_clean.csv and bike_day_clean.csv.
